# 🛡️ AI-Powered Credit Card Fraud Detection Platform
## Complete Data Science, ML Modeling & Explainability Notebook

**Author**: Lead AI & ML Engineering Team  
**Dataset**: Credit Card Fraud Detection (ULB Benchmark / Kaggle)  
**Models Benchmarked**: Logistic Regression, Random Forest, Cost-Sensitive XGBoost  
**Explainable AI (XAI)**: SHAP (SHapley Additive exPlanations) TreeExplainer  

### Step 1: Environment Setup & Library Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report
)
import xgboost as xgb
import shap

sns.set_theme(style="darkgrid")
print("Libraries successfully imported!")

### Step 2: Load Real Dataset (`creditcard.csv`)
- **Total Samples**: 284,807 transactions
- **Features**: `Time`, `Amount`, `V1`..`V28` (PCA transformed features)
- **Target**: `Class` (0: Legitimate, 1: Fraudulent)

In [ ]:
csv_path = os.path.join("data", "creditcard.csv")
if not os.path.exists(csv_path):
    csv_path = "creditcard.csv"

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")
df.head()

### Step 3: Exploratory Data Analysis (EDA)
- Missing values check
- Class imbalance inspection
- Summary statistics

In [ ]:
# Missing values check
missing = df.isnull().sum().sum()
print(f"Total Missing Values: {missing}")

# Class distribution
class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100
print(f"Legitimate (0): {class_counts[0]}")
print(f"Fraudulent (1): {class_counts[1]} ({fraud_pct:.3f}%)")

# Visualize class imbalance
plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#10b981", "#ef4444"])
plt.title("Extreme Class Imbalance (Legitimate vs Fraud)", fontsize=12, fontweight="bold")
plt.ylabel("Transaction Count")
plt.yscale("log")
plt.show()

### Step 4: Feature Preprocessing & Stratified Train/Test Split
- Apply `RobustScaler` to monetary `Amount` and `Time` (resilient to extreme outliers)
- Stratified K-Fold / Train-Test Split (80/20 ratio)

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test_scaled[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

### Step 5: Model Benchmarking (LR vs RF vs Cost-Sensitive XGBoost)
Optimizing for **Precision-Recall AUC (PR-AUC)**, the gold standard for severe class skew.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / max(1, (y_train == 1).sum())
print(f"Calculated XGBoost scale_pos_weight: {scale_pos_weight:.2f}")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    "XGBoost (Cost-Sensitive)": xgb.XGBClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.08,
        scale_pos_weight=scale_pos_weight, subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric="logloss"
    )
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = model.predict(X_test_scaled)
    
    pr_auc = average_precision_score(y_test, y_prob)
    roc_auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results[name] = {"PR-AUC": pr_auc, "ROC-AUC": roc_auc, "Precision": prec, "Recall": rec, "F1": f1}

pd.DataFrame(results).T.style.highlight_max(axis=0, color="#6366f1")

### Step 6: SHAP (SHapley Additive exPlanations) Explainability

In [ ]:
xgb_model = models["XGBoost (Cost-Sensitive)"]
explainer = shap.TreeExplainer(xgb_model)
sample_data = X_test_scaled.sample(n=300, random_state=42)
shap_values = explainer(sample_data)

# Global feature importance summary
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_data, show=False)
plt.title("Global SHAP Feature Importance Summary", fontsize=14, fontweight="bold")
plt.show()

### Step 7: Export Production Artifacts

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(xgb_model, os.path.join("models", "xgb_fraud_model.joblib"))
joblib.dump(scaler, os.path.join("models", "scaler.joblib"))
joblib.dump(explainer, os.path.join("models", "shap_explainer.joblib"))
print("Model, Scaler, and SHAP Explainer saved to models/ directory!")